In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_20_try_5.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['last_ones']
me:  20
trying: ['tqdm']
me:  0
trying: ['sample_submission']
me:  2
trying: ['myword']
me:  12
trying: ['train_first_last']
me:  11
trying: ['len_dict']
me:  12
trying: ['cols_to_display']
me:  14
trying: ['word_dict']
me:  12
trying: ['counter']
me:  11
trying: ['glob']
me:  0
trying: ['mylen']
me:  12
trying: ['ax']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['train_first']
me:  11
trying: ['train_last']
me:  11
trying: ['train']
me:  14
trying: ['stopwords']
me:  1
trying: ['plt']
me:  0
trying: ['add_gap_rows_2']
me:  22
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['FuncFormatter']
me:  0
trying: ['warnings']
me:  0
trying: ['pd']
me:  0
trying: ['cols_to_merge']
me:  13
trying: ['ax2']
me:  8
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['orig_output']
me:  21
trying: ['test_txt']
me:  1
trying: ['av_per_e

In [4]:
%%RecordEvent
%%time
### cell 21 ###

# lowercase the discourse text (runs on GPU via the cudf.pandas extension)
train['discourse_text'] = train['discourse_text'].str.lower()

# build the stopword list
stop_english = stopwords.words('english')
other_words_to_take_out = ['school', 'students', 'people', 'would', 'could', 'many']
stop_english.extend(other_words_to_take_out)

# split each discourse_text into words and explode so each word gets its own row
exploded = train.assign(Word=train['discourse_text'].str.split()).explode('Word')

# filter out stopwords (runs on GPU)
filtered = exploded[~exploded['Word'].isin(stop_english)]

# count frequencies of each word by discourse_type
freq = (
    filtered
    .groupby(['discourse_type', 'Word'], sort=False)
    .size()
    .reset_index(name='Frequency')
)

# for each discourse_type, take the top 10 words by Frequency
top10 = (
    freq
    .sort_values(['discourse_type', 'Frequency'], ascending=[True, False])
    .groupby('discourse_type', sort=False)
    .head(10)
)

# build counts_dict with pandas DataFrames so no custom class is required
dt_list = train['discourse_type'].unique().to_list()
counts_dict = {}
for dt in dt_list:
    sub = top10[top10['discourse_type'] == dt]
    df_small = (
        sub
        .drop('discourse_type', axis=1)
        .set_index('Word')
        .sort_values('Frequency', ascending=True)
        .to_pandas()
    )
    counts_dict[dt] = df_small

# extract keys in original order
keys = list(counts_dict.keys())

AttributeError: 'ndarray' object has no attribute 'to_list'

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_21_try_7.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_21_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[21], f)


In [ ]:
opt_output = Out.get(4)